In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sayan2807/yolo-training-dataset-annotated-soyabean")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/sayan2807/yolo-training-dataset-annotated-soyabean


In [ ]:
import shutil
import os

SOURCE_DIR = (
    "/kaggle/input/datasets/"
    "sayan2807/"
    "yolo-training-dataset-annotated-soyabean/"
    "Yolo seg format"
)

DEST_DIR = "/kaggle/working/dataset"

# Copy dataset
shutil.copytree(SOURCE_DIR, DEST_DIR, dirs_exist_ok=True)

print("Dataset copied.")

Dataset copied.


In [7]:
DATA_YAML = "/kaggle/working/dataset/data.yaml"

yaml_content = """
path: /kaggle/working/dataset

train: images/train
val: images/val

nc: 2

names:
  0: soy
  1: cotton
"""

with open(DATA_YAML, "w") as f:
    f.write(yaml_content)

print("data.yaml fixed.")

data.yaml fixed.


In [ ]:
# ============================================================
# INSTALL ULTRALYTICS
# ============================================================

!pip install -q ultralytics

# ============================================================
# IMPORTS
# ============================================================

import os
import shutil
import torch

from ultralytics import YOLO

# ============================================================
# GPU CHECK
# ============================================================

print("\n===================================================")
print("GPU CHECK")
print("===================================================")

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

else:
    raise RuntimeError("GPU NOT ENABLED")

# ============================================================
# ORIGINAL KAGGLE INPUT DATASET
# ============================================================

SOURCE_DIR = (
    "/kaggle/input/datasets/"
    "sayan2807/"
    "yolo-training-dataset-annotated-soyabean/"
    "Yolo seg format"
)

# ============================================================
# COPY DATASET TO WRITABLE DIRECTORY
# ============================================================

DEST_DIR = "/kaggle/working/dataset"

print("\nCopying dataset...")

shutil.copytree(SOURCE_DIR, DEST_DIR, dirs_exist_ok=True)

print("Dataset copied successfully.")

# ============================================================
# FIX DATA.YAML
# ============================================================

DATA_YAML = os.path.join(DEST_DIR, "data.yaml")

yaml_content = """
path: /kaggle/working/dataset

train: images/train
val: images/val

nc: 2

names:
  0: soy
  1: cotton
"""

with open(DATA_YAML, "w") as f:
    f.write(yaml_content)

print("\nFixed data.yaml")

# ============================================================
# VERIFY DATASET
# ============================================================

print("\n===================================================")
print("DATASET VERIFICATION")
print("===================================================")

print("\nDataset Contents:")
print(os.listdir(DEST_DIR))

print("\nTrain Images:", len(os.listdir(os.path.join(DEST_DIR, "images/train"))))

print("Validation Images:", len(os.listdir(os.path.join(DEST_DIR, "images/val"))))

print("Train Labels:", len(os.listdir(os.path.join(DEST_DIR, "labels/train"))))

print("Validation Labels:", len(os.listdir(os.path.join(DEST_DIR, "labels/val"))))

# ============================================================
# LOAD MODEL
# ============================================================

print("\n===================================================")
print("LOADING YOLO11 SEG MODEL")
print("===================================================")

model = YOLO("yolo11m-seg.pt")

print("Model loaded successfully.")

# ============================================================
# START TRAINING
# ============================================================

print("\n===================================================")
print("STARTING TRAINING")
print("===================================================\n")

results = model.train(
    # ========================================================
    # DATASET
    # ========================================================
    data=DATA_YAML,
    # ========================================================
    # TRAINING
    # ========================================================
    epochs=100,
    imgsz=640,
    batch=8,
    workers=4,
    device=0,
    pretrained=True,
    # ========================================================
    # OPTIMIZER
    # ========================================================
    optimizer="AdamW",
    lr0=0.001,
    patience=20,
    cos_lr=True,
    # ========================================================
    # AUGMENTATION
    # ========================================================
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.3,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    # ========================================================
    # SEGMENTATION SETTINGS
    # ========================================================
    overlap_mask=True,
    retina_masks=True,
    mask_ratio=4,
    # ========================================================
    # PERFORMANCE
    # ========================================================
    cache=True,
    amp=True,
    # ========================================================
    # OUTPUTS
    # ========================================================
    project="/kaggle/working",
    name="soycotton_yolo11_seg",
    save=True,
    plots=True,
    verbose=True,
)

# ============================================================
# FINAL OUTPUTS
# ============================================================

print("\n===================================================")
print("TRAINING COMPLETED")
print("===================================================")

BEST_MODEL = "/kaggle/working/soycotton_yolo11_seg/weights/best.pt"

LAST_MODEL = "/kaggle/working/soycotton_yolo11_seg/weights/last.pt"

print("\nBest model:")
print(BEST_MODEL)

print("\nLast model:")
print(LAST_MODEL)

print("\n===================================================")
print("OUTPUT DIRECTORY")
print("===================================================")

OUTPUT_DIR = "/kaggle/working/soycotton_yolo11_seg"

print(os.listdir(OUTPUT_DIR))


GPU CHECK
CUDA Available: True
GPU: Tesla T4

Copying dataset...
Dataset copied successfully.

Fixed data.yaml

DATASET VERIFICATION

Dataset Contents:
['images', 'labels', 'data.yaml']

Train Images: 512
Validation Images: 128
Train Labels: 512
Validation Labels: 128

LOADING YOLO11 SEG MODEL
Model loaded successfully.

STARTING TRAINING

Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/dataset/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h

# Inference

In [ ]:
import os
import cv2

from ultralytics import YOLO

# ============================================================
# CONFIG
# ============================================================

MODEL_PATH = r"/kaggle/working/soycotton_yolo11_seg-2/weights/best.pt"

IMAGE_PATH = (
    r"/kaggle/working/dataset/images/train/01b1f436-51c8-4dfa-9139-08dd809d251c.jpg"
)

OUTPUT_DIR = r"/kaggle/working/"

CONFIDENCE_THRESHOLD = 0.25

SAVE_CROPS = True

# ============================================================
# CREATE OUTPUT DIRECTORY
# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

if SAVE_CROPS:
    CROPS_DIR = os.path.join(OUTPUT_DIR, "leaf_crops")

    os.makedirs(CROPS_DIR, exist_ok=True)

# ============================================================
# LOAD MODEL
# ============================================================

print("\n===================================================")
print("LOADING YOLO11 MODEL")
print("===================================================")

model = YOLO(MODEL_PATH)

print("Model loaded successfully.")

# ============================================================
# LOAD IMAGE
# ============================================================

print("\nLoading UAV image...")

image = cv2.imread(IMAGE_PATH)

if image is None:
    raise ValueError("Failed to load UAV image.")

original_image = image.copy()

height, width = image.shape[:2]

print(f"Image Shape: {image.shape}")

# ============================================================
# RUN YOLO INFERENCE
# ============================================================

print("\nRunning inference...")

results = model.predict(
    source=image, conf=CONFIDENCE_THRESHOLD, save=False, verbose=False
)

result = results[0]

# ============================================================
# EXTRACT DETECTIONS
# ============================================================

boxes = result.boxes

leaf_count = 0

# ============================================================
# DRAW BOUNDING BOXES
# ============================================================

for idx, box in enumerate(boxes):
    confidence = float(box.conf[0])

    x1, y1, x2, y2 = box.xyxy[0]

    x1 = int(x1)
    y1 = int(y1)
    x2 = int(x2)
    y2 = int(y2)

    # ========================================================
    # DRAW BRIGHT RED BOUNDING BOX
    # ========================================================

    cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), 3)

    # ========================================================
    # LABEL
    # ========================================================

    label = f"Leaf {confidence:.2f}"

    cv2.putText(
        image, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2
    )

    leaf_count += 1

    # ========================================================
    # SAVE LEAF CROPS
    # ========================================================

    if SAVE_CROPS:
        crop = original_image[y1:y2, x1:x2]

        crop_path = os.path.join(CROPS_DIR, f"leaf_{idx + 1}.jpg")

        cv2.imwrite(crop_path, crop)

# ============================================================
# DISPLAY TOTAL LEAF COUNT
# ============================================================

count_text = f"Leaf Count: {leaf_count}"

cv2.putText(image, count_text, (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)

# ============================================================
# SAVE OUTPUT IMAGE
# ============================================================

output_image_path = os.path.join(OUTPUT_DIR, "uav_detected_leaves.jpg")

cv2.imwrite(output_image_path, image)

# ============================================================
# FINAL OUTPUT
# ============================================================

print("\n===================================================")
print("YOLO UAV INFERENCE COMPLETED")
print("===================================================")

print(f"\nDetected Leaves: {leaf_count}")

print(f"\nSaved Output Image:")
print(output_image_path)

if SAVE_CROPS:
    print(f"\nSaved Leaf Crops:")
    print(CROPS_DIR)

# ============================================================
# SHOW RESULT
# ============================================================

cv2.imshow("YOLO UAV Leaf Detection", image)

cv2.waitKey(0)

cv2.destroyAllWindows()


LOADING YOLO11 MODEL
Model loaded successfully.

Loading UAV image...
Image Shape: (1600, 1200, 3)

Running inference...
